In [2]:
from transformers import AutoTokenizer, AutoConfig
from bertviz.transformers_neuron_view import BertModel
from bertviz.neuron_view import show
import torch
from torch import nn
import torch.nn.functional as F
from math import sqrt

In [3]:
MODEL = "bert-base-uncased"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = BertModel.from_pretrained(MODEL)

In [7]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): BertLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): BertLayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): BertIntermediate(
          (den

In [12]:
text = "io sono gabriele e mi piace il sushi"
show(model, "bert", tokenizer, text)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
inputs = tokenizer(text, return_tensors="pt", add_special_tokens= False)

In [18]:
config = AutoConfig.from_pretrained(MODEL)

In [20]:
token_emb = nn.Embedding(config.vocab_size, config.hidden_size)

In [24]:
embeddings = token_emb(inputs.input_ids)

In [26]:
class AttentionHead(nn.Module):

    def scaled_dot_product_attn(self, q, k, v):
        dim_k = q.size(-1)
        scores = torch.bmm(q, k.transpose(1,2)) / sqrt(dim_k)
        weights = F.softmax(scores, dim=-1)
        attention = torch.bmm(weights, v)

        return attention

    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.q = nn.Linear(embed_dim, head_dim)
        self.k = nn.Linear(embed_dim, head_dim)
        self.v = nn.Linear(embed_dim, head_dim)

    def forward(self, hidden):
        attn_outputs = self.scaled_dot_product_attn(
            self.q(hidden), self.k(hidden), self.v(hidden)
        )
        return attn_outputs

In [28]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        embed_dim = config.hidden_size
        num_heads = config.num_attention_heads
        head_dim = embed_dim // num_heads

        self.heads = nn.ModuleList(
            [AttentionHead(embed_dim, head_dim) for _ in range(num_heads)]
        )

        self.output_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, hidden):
        x = torch.cat([h(hidden) for h in self.heads], dim=-1)
        x = self.output_linear(x)
        return x

In [29]:
multihead_attn = MultiHeadAttention(config)
attn_output = multihead_attn(embeddings)

In [32]:
attn_output.size()

torch.Size([1, 12, 768])